# Predict Irrigation Need - Kaggle Submission Notebook

This notebook reproduces the repo pipeline in a Kaggle-friendly format and writes a submission file for `playground-series-s6e4`.

- Competition data source: `/kaggle/input/playground-series-s6e4`
- Final output: `/kaggle/working/submission.csv`
- Runtime note: this is a full training pipeline and can take significant time.

In [ ]:
!pip -q install xgboost lightgbm catboost optuna pyarrow

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/VedantPol/predict-irrigation-kaggle-solution.git"
WORKDIR = Path("/kaggle/working/predict-irrigation-kaggle-solution")
COMP_DIR = Path("/kaggle/input/playground-series-s6e4")

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)

data_dir = WORKDIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

for file_name in ["train.csv", "test.csv", "sample_submission.csv"]:
    src = COMP_DIR / file_name
    if src.exists():
        shutil.copy2(src, data_dir / file_name)

sys.path.insert(0, str(WORKDIR))
print("Repo ready at:", WORKDIR)
print("Data files:", sorted([p.name for p in data_dir.glob("*.csv")]))

In [ ]:
from scripts.run_pipeline import PREPARE_DATA_CONFIG, FEATURE_ENGINEERING_CONFIG, BASELINE_CONFIG
from src.fraud_risk_early_warning.pipeline import run_stage_with_config

prepare_cfg = dict(PREPARE_DATA_CONFIG)
feature_cfg = dict(FEATURE_ENGINEERING_CONFIG)
baseline_cfg = dict(BASELINE_CONFIG)

# Kaggle-public safe defaults: CPU-compatible tree suite without RAPIDS dependencies.
baseline_cfg.update(
    {
        "strict_gpu_only": False,
        "tree_libraries": ("xgboost", "catboost", "lightgbm"),
        "use_tree_optuna": False,
        "tree_optuna_trials_per_library": 0,
        "run_deep_level2_stack": False,
        "models_per_library": 4,
    }
)

STAGES = [
    "prepare_data",
    "feature_engineering",
    "baseline",
    "eda",
    "hill_climb",
    "stacking",
    "final_blend",
]

In [ ]:
for stage in STAGES:
    print(f"\\n=== Running stage: {stage} ===")
    run_stage_with_config(
        stage=stage,
        prepare_data_config=prepare_cfg,
        feature_engineering_config=feature_cfg,
        baseline_config=baseline_cfg,
    )

print("\\nPipeline run complete.")

In [ ]:
import pandas as pd

submission_path = WORKDIR / "outputs" / "submissions" / "submission_final_blend.csv"
submission = pd.read_csv(submission_path)
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission saved to /kaggle/working/submission.csv")
submission.head()

## Submit and Share Publicly

1. Click **Save Version** and select **Save & Run All (Commit)**.
2. After completion, click **Submit to Competition** and choose `/kaggle/working/submission.csv`.
3. Open notebook **Settings** and set **Visibility** to **Public**.
4. Share the notebook URL so others can view, copy, run, and edit their own version.